Крок 1. Завантаження підготовлених даних
Відкрий 03_model_training.ipynb у VS Code та встав у першу комірку:

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/Users/glebsavelev/Desktop/da/results/data")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze("columns")

print("Навчальна вибірка:", X_train.shape)
print("Тестова вибірка:", X_test.shape)

print("\nРозподіл класів у навчальній вибірці:")
print(y_train.value_counts())

print("\nРозподіл класів у тестовій вибірці:")
print(y_test.value_counts())

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert list(X_train.columns) == list(X_test.columns)

print("\nПеревірка даних успішна.")

Навчальна вибірка: (76916, 8)
Тестова вибірка: (19230, 8)

Розподіл класів у навчальній вибірці:
diabetes
0    70130
1     6786
Name: count, dtype: int64

Розподіл класів у тестовій вибірці:
diabetes
0    17534
1     1696
Name: count, dtype: int64

Перевірка даних успішна.


Крок 2 — створення базової Logistic Regression
Потрібно повторно налаштувати попередню обробку, оскільки змінні з 02_preprocessing.ipynb автоматично не передаються в інший ноутбук.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Категоріальні та числові ознаки
categorical_features = ["gender", "smoking_history"]

numeric_features = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

# Обробка числових ознак
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Обробка категоріальних ознак
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Об'єднання перетворень
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

# Базова модель без корекції дисбалансу класів
lr_baseline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print("Базову модель Logistic Regression створено.")
print("Попередня обробка включена до Pipeline.")
print("Модель ще не навчена.")

Базову модель Logistic Regression створено.
Попередня обробка включена до Pipeline.
Модель ще не навчена.


Крок 3. Навчання Logistic Regression
Оскільки в нашому датасеті лише 8,82% записів належать до позитивного класу, важливо перевірити, наскільки добре модель виявляє саме їх.
Для першого експерименту використаємо внутрішню валідаційну вибірку з навчальних даних. Основну тестову вибірку поки не чіпаємо.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Ділимо навчальну вибірку на:
# 80% для навчання та 20% для валідації
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

# Навчаємо модель лише на X_fit
lr_baseline.fit(X_fit, y_fit)

# Отримуємо прогнози для валідаційної вибірки
y_pred = lr_baseline.predict(X_val)
y_proba = lr_baseline.predict_proba(X_val)[:, 1]

# Обчислюємо метрики
print("=== Logistic Regression: Baseline ===")

print("Accuracy:", round(accuracy_score(y_val, y_pred), 4))
print("Precision:", round(precision_score(y_val, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_val, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_proba), 4))
print("PR-AUC:", round(average_precision_score(y_val, y_proba), 4))

print("\nМатриця помилок:")
print(confusion_matrix(y_val, y_pred))

print("\nРозмір навчальної частини:", X_fit.shape)
print("Розмір валідаційної частини:", X_val.shape)

=== Logistic Regression: Baseline ===
Accuracy: 0.9583
Precision: 0.8623
Recall: 0.6279
F1-score: 0.7267
ROC-AUC: 0.9599
PR-AUC: 0.8135

Матриця помилок:
[[13891   136]
 [  505   852]]

Розмір навчальної частини: (61532, 8)
Розмір валідаційної частини: (15384, 8)


Метрика	Що показує
Accuracy	Частка всіх правильних прогнозів
Precision	Яка частка прогнозованих випадків діабету справді належить до класу 1
Recall	Яку частку випадків діабету модель змогла виявити
F1-score	Баланс між Precision та Recall
ROC-AUC	Здатність моделі розрізняти класи за різних порогів
PR-AUC	Якість моделі за співвідношенням Precision і Recall при різних порогах

Крок 4 Перевіримо, чи допоможе врахування дисбалансу зменшити кількість хибнонегативних прогнозів.
class_weight="balanced" надає більшу вагу помилкам на менш представленому класі під час навчання.

In [4]:
from sklearn.base import clone

# Створюємо незалежну копію базової моделі
lr_balanced = clone(lr_baseline)

# Увімкнення балансування класів
lr_balanced.set_params(
    classifier__class_weight="balanced"
)

# Навчання на тій самій навчальній частині
lr_balanced.fit(X_fit, y_fit)

# Прогнози на тій самій валідаційній частині
y_pred_balanced = lr_balanced.predict(X_val)
y_proba_balanced = lr_balanced.predict_proba(X_val)[:, 1]

print("=== Logistic Regression: Balanced ===")

print("Accuracy:", round(accuracy_score(y_val, y_pred_balanced), 4))
print("Precision:", round(precision_score(y_val, y_pred_balanced), 4))
print("Recall:", round(recall_score(y_val, y_pred_balanced), 4))
print("F1-score:", round(f1_score(y_val, y_pred_balanced), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_proba_balanced), 4))
print("PR-AUC:", round(average_precision_score(y_val, y_proba_balanced), 4))

print("\nМатриця помилок:")
print(confusion_matrix(y_val, y_pred_balanced))

=== Logistic Regression: Balanced ===
Accuracy: 0.8823
Precision: 0.4202
Recall: 0.8784
F1-score: 0.5684
ROC-AUC: 0.9603
PR-AUC: 0.8123

Матриця помилок:
[[12382  1645]
 [  165  1192]]


Крок 5 — дослідження порогу класифікації
За замовчуванням Logistic Regression використовує поріг 0,5.
Тепер перевіримо, чи можна знайти кращий баланс між Precision і Recall без повторного навчання базової моделі.
Дослідимо пороги від 0,10 до 0,90 на валідаційній вибірці. Для початку критерієм вибору буде максимальне значення F1-score.

In [5]:
import numpy as np
import pandas as pd

from sklearn.metrics import precision_score, recall_score, f1_score

# Ймовірності базової Logistic Regression,
# отримані раніше на валідаційній вибірці
thresholds = np.arange(0.10, 0.91, 0.05)

results = []

for threshold in thresholds:
    predictions = (y_proba >= threshold).astype(int)

    results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "FN": int(((y_val == 1) & (predictions == 0)).sum()),
        "FP": int(((y_val == 0) & (predictions == 1)).sum())
    })

threshold_results = pd.DataFrame(results)

# Вибір порогу за максимальним F1-score
best_idx = threshold_results["F1"].idxmax()
best_row = threshold_results.loc[best_idx]

best_threshold = float(best_row["Threshold"])

print("=== Результати дослідження порогів ===")
print(threshold_results.round(4).to_string(index=False))

print("\n=== Найкращий поріг за F1-score ===")
print("Threshold:", best_threshold)
print("Precision:", round(best_row["Precision"], 4))
print("Recall:", round(best_row["Recall"], 4))
print("F1-score:", round(best_row["F1"], 4))
print("FN:", int(best_row["FN"]))
print("FP:", int(best_row["FP"]))

=== Результати дослідження порогів ===
 Threshold  Precision  Recall     F1  FN   FP
      0.10     0.4451  0.8578 0.5861 193 1451
      0.15     0.5222  0.8143 0.6363 252 1011
      0.20     0.5900  0.7775 0.6709 302  733
      0.25     0.6507  0.7494 0.6966 340  546
      0.30     0.7105  0.7251 0.7177 373  401
      0.35     0.7504  0.6979 0.7232 410  315
      0.40     0.7928  0.6794 0.7317 435  241
      0.45     0.8254  0.6514 0.7282 473  187
      0.50     0.8623  0.6279 0.7267 505  136
      0.55     0.8827  0.6043 0.7174 537  109
      0.60     0.9021  0.5836 0.7087 565   86
      0.65     0.9207  0.5564 0.6936 602   65
      0.70     0.9383  0.5380 0.6838 627   48
      0.75     0.9503  0.5070 0.6612 669   36
      0.80     0.9731  0.4805 0.6433 705   18
      0.85     0.9903  0.4503 0.6190 746    6
      0.90     0.9982  0.4141 0.5854 795    1

=== Найкращий поріг за F1-score ===
Threshold: 0.4
Precision: 0.7928
Recall: 0.6794
F1-score: 0.7317
FN: 435
FP: 241


Крок 6 — поріг із вимогою Recall ≥ 80%
Для нашого дослідження цікаво перевірити ще один підхід: вибирати поріг не за максимальним F1-score, а за умовою виявлення щонайменше 80% позитивних записів.
Серед порогів, які забезпечують Recall ≥ 80%, виберемо той, що має найбільший Precision.
Це експериментальний критерій, а не медичний стандарт.

In [6]:
# Мінімально допустимий Recall для експерименту
target_recall = 0.80

# Відбираємо пороги, які задовольняють умову
eligible = threshold_results[
    threshold_results["Recall"] >= target_recall
].copy()

if eligible.empty:
    print("Не знайдено порогів із Recall >= 0.80")
else:
    # Максимальний Precision серед допустимих порогів
    selected = eligible.sort_values(
        by=["Precision", "F1"],
        ascending=False
    ).iloc[0]

    recall_threshold = float(selected["Threshold"])

    print("=== Вибір порогу за умовою Recall >= 80% ===")
    print("Threshold:", recall_threshold)
    print("Precision:", round(selected["Precision"], 4))
    print("Recall:", round(selected["Recall"], 4))
    print("F1-score:", round(selected["F1"], 4))
    print("FN:", int(selected["FN"]))
    print("FP:", int(selected["FP"]))

=== Вибір порогу за умовою Recall >= 80% ===
Threshold: 0.15
Precision: 0.5222
Recall: 0.8143
F1-score: 0.6363
FN: 252
FP: 1011


In [ ]:
Крок 7 — навчання Random Forest
Тепер перевіримо другий алгоритм — Random Forest (випадковий ліс).
Він використовує ансамбль дерев рішень і 
здатний враховувати нелінійні залежності між ознаками.
Почнемо з базового варіанта без балансування класів. 
Для чесного порівняння використаємо ті самі X_fit, X_val, y_fit, y_val.

n_estimators=100	Кількість дерев у лісі
max_depth=12	Максимальна глибина кожного дерева
min_samples_leaf=2	Мінімальна кількість записів у листі дерева
random_state=42	Відтворюваність результатів
n_jobs=-1	Використання доступних процесорних ядер

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

# Створюємо базову модель Random Forest
rf_baseline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# Навчання лише на навчальній частині
rf_baseline.fit(X_fit, y_fit)

# Прогнози на валідаційній частині
y_pred_rf = rf_baseline.predict(X_val)
y_proba_rf = rf_baseline.predict_proba(X_val)[:, 1]

print("=== Random Forest: Baseline ===")

print("Accuracy:", round(accuracy_score(y_val, y_pred_rf), 4))
print("Precision:", round(precision_score(y_val, y_pred_rf, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, y_pred_rf, zero_division=0), 4))
print("F1-score:", round(f1_score(y_val, y_pred_rf, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_proba_rf), 4))
print("PR-AUC:", round(average_precision_score(y_val, y_proba_rf), 4))

print("\nМатриця помилок:")
print(confusion_matrix(y_val, y_pred_rf))

=== Random Forest: Baseline ===
Accuracy: 0.9707
Precision: 0.9956
Recall: 0.6713
F1-score: 0.8019
ROC-AUC: 0.9709
PR-AUC: 0.8743

Матриця помилок:
[[14023     4]
 [  446   911]]


Крок 8 — Random Forest із балансуванням класів
Тепер перевіримо, як зміняться результати Random Forest, якщо додати class_weight="balanced".
Для порівняння використовуємо ті самі навчальну та валідаційну вибірки.
Додай восьму комірку у 03_model_training.ipynb:

In [8]:
from sklearn.base import clone

# Створюємо незалежну копію Random Forest
rf_balanced = clone(rf_baseline)

# Додаємо балансування класів
rf_balanced.set_params(
    classifier__class_weight="balanced"
)

# Навчаємо на тій самій навчальній частині
rf_balanced.fit(X_fit, y_fit)

# Прогнози для валідаційної вибірки
y_pred_rf_balanced = rf_balanced.predict(X_val)
y_proba_rf_balanced = rf_balanced.predict_proba(X_val)[:, 1]

print("=== Random Forest: Balanced ===")

print("Accuracy:", round(
    accuracy_score(y_val, y_pred_rf_balanced), 4
))
print("Precision:", round(
    precision_score(y_val, y_pred_rf_balanced, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_val, y_pred_rf_balanced, zero_division=0), 4
))
print("F1-score:", round(
    f1_score(y_val, y_pred_rf_balanced, zero_division=0), 4
))
print("ROC-AUC:", round(
    roc_auc_score(y_val, y_proba_rf_balanced), 4
))
print("PR-AUC:", round(
    average_precision_score(y_val, y_proba_rf_balanced), 4
))

print("\nМатриця помилок:")
print(confusion_matrix(y_val, y_pred_rf_balanced))

=== Random Forest: Balanced ===
Accuracy: 0.9129
Precision: 0.5035
Recall: 0.8983
F1-score: 0.6453
ROC-AUC: 0.9759
PR-AUC: 0.8796

Матриця помилок:
[[12825  1202]
 [  138  1219]]


Крок 9 — оптимізація порогу для Random Forest
У Logistic Regression ми вже дослідили пороги класифікації. Тепер зробимо аналогічний експеримент для базового Random Forest.
Перевіримо пороги від 0,10 до 0,90 з кроком 0,05 та знайдемо:
- поріг із найбільшим F1-score;
- поріг із найбільшим Precision за умови Recall ≥ 80%.
Так ми зможемо порівняти балансування класів зі зміною порогу без повторного навчання моделі.

In [9]:
# Дослідження порогів базової моделі Random Forest
rf_thresholds = np.arange(0.10, 0.91, 0.05)

rf_results = []

for threshold in rf_thresholds:
    predictions = (y_proba_rf >= threshold).astype(int)

    rf_results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "FN": int(((y_val == 1) & (predictions == 0)).sum()),
        "FP": int(((y_val == 0) & (predictions == 1)).sum())
    })

rf_threshold_results = pd.DataFrame(rf_results)

print("=== Random Forest: дослідження порогів ===")
print(rf_threshold_results.round(4).to_string(index=False))

# Поріг за максимальним F1-score
rf_best = rf_threshold_results.loc[
    rf_threshold_results["F1"].idxmax()
]
rf_best_threshold = float(rf_best["Threshold"])

print("\n=== Найкращий поріг за F1-score ===")
print(rf_best.to_string())

# Поріг за вимогою Recall >= 80%
rf_eligible = rf_threshold_results[
    rf_threshold_results["Recall"] >= 0.80
]

if not rf_eligible.empty:
    rf_selected = rf_eligible.sort_values(
        by=["Precision", "F1"],
        ascending=False
    ).iloc[0]

    rf_recall_threshold = float(rf_selected["Threshold"])

    print("\n=== Найкращий Precision при Recall >= 80% ===")
    print(rf_selected.to_string())
else:
    print("\nПоріг із Recall >= 80% не знайдено.")

=== Random Forest: дослідження порогів ===
 Threshold  Precision  Recall     F1  FN   FP
      0.10     0.5228  0.8784 0.6555 165 1088
      0.15     0.6673  0.8128 0.7329 254  550
      0.20     0.7641  0.7686 0.7663 314  322
      0.25     0.8646  0.7340 0.7939 361  156
      0.30     0.9403  0.7082 0.8079 396   61
      0.35     0.9679  0.6890 0.8050 422   31
      0.40     0.9809  0.6802 0.8033 434   18
      0.45     0.9913  0.6750 0.8032 441    8
      0.50     0.9956  0.6713 0.8019 446    4
      0.55     1.0000  0.6699 0.8023 448    0
      0.60     1.0000  0.6691 0.8018 449    0
      0.65     1.0000  0.6691 0.8018 449    0
      0.70     1.0000  0.6691 0.8018 449    0
      0.75     1.0000  0.6691 0.8018 449    0
      0.80     1.0000  0.6684 0.8012 450    0
      0.85     1.0000  0.6662 0.7996 453    0
      0.90     1.0000  0.6205 0.7658 515    0

=== Найкращий поріг за F1-score ===
Threshold      0.300000
Precision      0.940313
Recall         0.708180
F1             0.807

Крок 10 — створення третьої моделі SVM
Тепер переходимо до третього алгоритму — Support Vector Machine (метод опорних векторів).
SVM визначає межу між класами. Для наших експериментів використаємо лінійний SVM (LinearSVC), оскільки повний SVM із нелінійним RBF-ядром може навчатися досить довго на 60 тисячах записів.
Лінійний варіант дозволить швидше отримати відтворювані результати на твоєму Mac.
Додай десяту комірку до 03_model_training.ipynb

In [10]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone

# Лінійний SVM з калібруванням оцінок ймовірності
svm_baseline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", CalibratedClassifierCV(
        estimator=LinearSVC(
            C=1.0,
            max_iter=5000,
            random_state=42
        ),
        method="sigmoid",
        cv=3
    ))
])

print("Модель SVM створено.")
print("Тип: LinearSVC + калібрування ймовірностей.")
print("Модель ще не навчена.")

Модель SVM створено.
Тип: LinearSVC + калібрування ймовірностей.
Модель ще не навчена.


Крок 11. Навчання базової моделі SVM
Використаємо ті самі X_fit, y_fit, X_val та y_val, що й для попередніх моделей. Це забезпечить порівнянність результатів.

In [11]:
# Навчаємо базову модель SVM
svm_baseline.fit(X_fit, y_fit)

# Отримуємо прогнози на валідаційній вибірці
y_pred_svm = svm_baseline.predict(X_val)
y_proba_svm = svm_baseline.predict_proba(X_val)[:, 1]

print("=== SVM: Baseline ===")

print("Accuracy:", round(
    accuracy_score(y_val, y_pred_svm), 4
))

print("Precision:", round(
    precision_score(y_val, y_pred_svm, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_val, y_pred_svm, zero_division=0), 4
))

print("F1-score:", round(
    f1_score(y_val, y_pred_svm, zero_division=0), 4
))

print("ROC-AUC:", round(
    roc_auc_score(y_val, y_proba_svm), 4
))

print("PR-AUC:", round(
    average_precision_score(y_val, y_proba_svm), 4
))

print("\nМатриця помилок:")
print(confusion_matrix(y_val, y_pred_svm))

=== SVM: Baseline ===
Accuracy: 0.9582
Precision: 0.8621
Recall: 0.6264
F1-score: 0.7256
ROC-AUC: 0.9594
PR-AUC: 0.8131

Матриця помилок:
[[13891   136]
 [  507   850]]


Крок 12 — SVM із балансуванням класів
Тепер перевіримо, чи допоможе class_weight="balanced" зменшити кількість пропущених позитивних записів для SVM.
Ми вже побачили, що для Logistic Regression і Random Forest балансування суттєво збільшувало Recall. Тепер перевіримо цю закономірність на третьому алгоритмі.

In [12]:
from sklearn.base import clone

# Створюємо незалежну копію базової моделі SVM
svm_balanced = clone(svm_baseline)

# Увімкнення балансування класів у LinearSVC
svm_balanced.set_params(
    classifier__estimator__class_weight="balanced"
)

# Навчання на тій самій навчальній частині
svm_balanced.fit(X_fit, y_fit)

# Прогнози на валідаційній вибірці
y_pred_svm_balanced = svm_balanced.predict(X_val)
y_proba_svm_balanced = svm_balanced.predict_proba(X_val)[:, 1]

print("=== SVM: Balanced ===")

print("Accuracy:", round(
    accuracy_score(y_val, y_pred_svm_balanced), 4
))
print("Precision:", round(
    precision_score(y_val, y_pred_svm_balanced, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_val, y_pred_svm_balanced, zero_division=0), 4
))
print("F1-score:", round(
    f1_score(y_val, y_pred_svm_balanced, zero_division=0), 4
))
print("ROC-AUC:", round(
    roc_auc_score(y_val, y_proba_svm_balanced), 4
))
print("PR-AUC:", round(
    average_precision_score(y_val, y_proba_svm_balanced), 4
))

print("\nМатриця помилок:")
print(confusion_matrix(y_val, y_pred_svm_balanced))

=== SVM: Balanced ===
Accuracy: 0.9576
Precision: 0.8492
Recall: 0.6308
F1-score: 0.7239
ROC-AUC: 0.9603
PR-AUC: 0.8115

Матриця помилок:
[[13875   152]
 [  501   856]]


Крок 13 — дослідження порогів SVM
Тепер завершуємо порівняння трьох алгоритмів. Для базового SVM перевіримо пороги класифікації так само, як раніше для Logistic Regression і Random Forest.

In [13]:
# Дослідження порогів базової моделі SVM
svm_thresholds = np.arange(0.10, 0.91, 0.05)

svm_results = []

for threshold in svm_thresholds:
    predictions = (y_proba_svm >= threshold).astype(int)

    svm_results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "FN": int(((y_val == 1) & (predictions == 0)).sum()),
        "FP": int(((y_val == 0) & (predictions == 1)).sum())
    })

svm_threshold_results = pd.DataFrame(svm_results)

print("=== SVM: дослідження порогів ===")
print(svm_threshold_results.round(4).to_string(index=False))

# Поріг із максимальним F1-score
svm_best = svm_threshold_results.loc[
    svm_threshold_results["F1"].idxmax()
]

svm_best_threshold = float(svm_best["Threshold"])

print("\n=== Найкращий поріг за F1-score ===")
print(svm_best.to_string())

# Поріг із Recall не нижче 80%
svm_eligible = svm_threshold_results[
    svm_threshold_results["Recall"] >= 0.80
]

if not svm_eligible.empty:
    svm_selected = svm_eligible.sort_values(
        by=["Precision", "F1"],
        ascending=False
    ).iloc[0]

    svm_recall_threshold = float(svm_selected["Threshold"])

    print("\n=== Найкращий Precision при Recall >= 80% ===")
    print(svm_selected.to_string())
else:
    print("\nПоріг із Recall >= 80% не знайдено.")

=== SVM: дослідження порогів ===
 Threshold  Precision  Recall     F1  FN   FP
      0.10     0.4445  0.8497 0.5836 204 1441
      0.15     0.5259  0.8150 0.6393 251  997
      0.20     0.5904  0.7745 0.6701 306  729
      0.25     0.6567  0.7472 0.6991 343  530
      0.30     0.7161  0.7192 0.7176 381  387
      0.35     0.7520  0.6949 0.7223 414  311
      0.40     0.8002  0.6758 0.7327 440  229
      0.45     0.8365  0.6485 0.7306 477  172
      0.50     0.8621  0.6264 0.7256 507  136
      0.55     0.8831  0.6013 0.7155 541  108
      0.60     0.8999  0.5829 0.7075 566   88
      0.65     0.9212  0.5601 0.6966 597   65
      0.70     0.9366  0.5335 0.6798 633   49
      0.75     0.9559  0.5107 0.6657 664   32
      0.80     0.9719  0.4834 0.6457 701   19
      0.85     0.9903  0.4517 0.6204 744    6
      0.90     0.9983  0.4208 0.5920 786    1

=== Найкращий поріг за F1-score ===
Threshold      0.400000
Precision      0.800175
Recall         0.675755
F1             0.732721
FN    

Крок 14 — збереження результатів експериментів
Зараз завершимо роботу у файлі 03_model_training.ipynb.
Нам потрібно зберегти таблиці досліджених порогів і короткий підсумок, щоб використати їх у четвертому розділі магістерської.

In [14]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/Users/glebsavelev/Desktop/da/results/tables"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Зберігаємо результати дослідження порогів
threshold_results.to_csv(
    OUTPUT_DIR / "logistic_regression_thresholds.csv",
    index=False
)

rf_threshold_results.to_csv(
    OUTPUT_DIR / "random_forest_thresholds.csv",
    index=False
)

svm_threshold_results.to_csv(
    OUTPUT_DIR / "svm_thresholds.csv",
    index=False
)

# Підсумкова таблиця відібраних конфігурацій
summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Criterion": "Max F1",
        "Threshold": best_threshold,
        "F1": best_row["F1"],
        "Recall": best_row["Recall"],
        "Precision": best_row["Precision"]
    },
    {
        "Model": "Random Forest",
        "Criterion": "Max F1",
        "Threshold": rf_best_threshold,
        "F1": rf_best["F1"],
        "Recall": rf_best["Recall"],
        "Precision": rf_best["Precision"]
    },
    {
        "Model": "SVM",
        "Criterion": "Max F1",
        "Threshold": svm_best_threshold,
        "F1": svm_best["F1"],
        "Recall": svm_best["Recall"],
        "Precision": svm_best["Precision"]
    }
])

summary.to_csv(
    OUTPUT_DIR / "model_validation_summary.csv",
    index=False
)

print("Результати експериментів збережено!")
print("\nПідсумкова таблиця:")
print(summary.round(4).to_string(index=False))

print("\nЗбережені файли:")
for file in sorted(OUTPUT_DIR.glob("*thresholds.csv")):
    print("-", file.name)

print("-", "model_validation_summary.csv")

Результати експериментів збережено!

Підсумкова таблиця:
              Model Criterion  Threshold     F1  Recall  Precision
Logistic Regression    Max F1        0.4 0.7317  0.6794     0.7928
      Random Forest    Max F1        0.3 0.8079  0.7082     0.9403
                SVM    Max F1        0.4 0.7327  0.6758     0.8002

Збережені файли:
- logistic_regression_thresholds.csv
- random_forest_thresholds.csv
- svm_thresholds.csv
- model_validation_summary.csv
